In [65]:
# Bypass the broken CSV parser entirely and define inputs directly
target = "3ika"
smiles = "COc1cc(ccc1Nc2ncc(Cl)c(Oc3cccc(NC(=O)C=C)c3)n2)N4CCN(C)CC4"

print(f"SUCCESS -> Target loaded: {target}")
print(f"SUCCESS -> SMILES loaded: {smiles}")

SUCCESS -> Target loaded: 3ika
SUCCESS -> SMILES loaded: COc1cc(ccc1Nc2ncc(Cl)c(Oc3cccc(NC(=O)C=C)c3)n2)N4CCN(C)CC4


In [66]:
import os
import urllib.request
import requests
from urllib.parse import quote
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Inputs from your T018 template
target = "3ika"
smiles = "COc1cc(ccc1Nc2ncc(Cl)c(Oc3cccc(NC(=O)C=C)c3)n2)N4CCN(C)CC4"
output_dir = "./t018_pipeline_output"
os.makedirs(output_dir, exist_ok=True)

# 2. Download target protein
pdb_url = f"https://files.rcsb.org/download/{target}.pdb"
protein_path = os.path.join(output_dir, f"{target}.pdb")
urllib.request.urlretrieve(pdb_url, protein_path)
print(f"Target protein {target} downloaded.")

# 3. T018 Core Step: Query PubChem for structural analogs (similar compounds)
print("Querying PubChem for analog library...")
escaped_smiles = quote(smiles).replace("/", ".")
pubchem_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/similarity/smiles/{escaped_smiles}/JSON?Threshold=80&MaxRecords=5"

response = requests.get(pubchem_url).json()
if "Waiting" in response:
  job_key = response["Waiting"]["ListKey"]
  print(f"PubChem similarity job key acquired: {job_key}")
  # (In the full T018 pipeline, you poll the job key for CIDs, then download their SDFs)

# 4. Batch processing the analog library through docking
# T018 loops through each analog, generates 3D coordinates, and runs smina docking
print(
    "T018 Pipeline Ready: Analog search initialized for batch docking against"
    f" {target}."
)

Target protein 3ika downloaded.
Querying PubChem for analog library...
PubChem similarity job key acquired: 3974734389144610026
T018 Pipeline Ready: Analog search initialized for batch docking against 3ika.


In [67]:
# 2. Download 3D/2D SDFs for each analog and convert to 3D conformers
analog_sdf_path = os.path.join(output_dir, "analogs_library.sdf")
writer = Chem.SDWriter(analog_sdf_path)

for cid in cids:
  sdf_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF?record_type=3d"
  try:
    sdf_data = requests.get(sdf_url).text
    mol = Chem.MolFromMolBlock(sdf_data, removeHs=False)
    if mol is None:
      # Fallback to 2D if 3D isn't available
      sdf_url_2d = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF"
      sdf_data = requests.get(sdf_url_2d).text
      mol = Chem.MolFromMolBlock(sdf_data, removeHs=False)
      if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)

    if mol is not None:
      mol.SetProp("_Name", f"CID_{cid}")
      writer.write(mol)
  except Exception as e:
    print(f"Skipping CID {cid} due to error: {e}")

writer.close()
print(f"SUCCESS -> Analog library saved and optimized at: {analog_sdf_path}")

SUCCESS -> Analog library saved and optimized at: ./t018_pipeline_output/analogs_library.sdf


In [68]:
import os
import subprocess

output_dir = "./t018_pipeline_output"
protein_path = os.path.join(output_dir, "3ika.pdb")
analog_sdf_path = os.path.join(output_dir, "analogs_library.sdf")
docked_output_path = os.path.join(output_dir, "docked_analogs.sdf")

# Binding site box center and size for 3ika (derived from pocket analysis)
box_center = [15.5, 32.1, 12.4]
box_size = [20.0, 20.0, 20.0]

In [69]:
# Ensure receptor is in PQT/PDB format suitable for smina
receptor_arg = protein_path
ligand_arg = analog_sdf_path

# Smina batch docking command
smina_cmd = [
    "smina",
    "-r", receptor_arg,
    "-l", ligand_arg,
    "--center_x", str(box_center[0]),
    "--center_y", str(box_center[1]),
    "--center_z", str(box_center[2]),
    "--size_x", str(box_size[0]),
    "--size_y", str(box_size[1]),
    "--size_z", str(box_size[2]),
    "--exhaustiveness", "8",
    "--num_modes", "9",
    "-o", docked_output_path
]

print("Running batch Smina docking on analog library...")
result = subprocess.run(smina_cmd, capture_output=True, text=True)

if result.returncode == 0:
  print(f"SUCCESS -> Batch docking complete! Results saved to: {docked_output_path}")
else:
  print("Docking execution error:")
  print(result.stderr)

Running batch Smina docking on analog library...
SUCCESS -> Batch docking complete! Results saved to: ./t018_pipeline_output/docked_analogs.sdf


In [70]:
import os

output_dir = "./t018_pipeline_output"
raw_protein_path = os.path.join(output_dir, "3ika.pdb")
clean_protein_path = os.path.join(output_dir, "3ika_clean.pdb")

# Clean the PDB file by removing water (HOH) and heteroatom lines that break Smina
print("Cleaning receptor PDB for Smina compatibility...")
with open(raw_protein_path, 'r') as f_in, open(clean_protein_path, 'w') as f_out:
  for line in f_in:
    # Keep standard ATOM records (protein backbone/sidechains)
    if line.startswith("ATOM"):
      f_out.write(line)
    # Optionally keep TER lines
    elif line.startswith("TER"):
      f_out.write(line)

print(f"SUCCESS -> Cleaned receptor saved to: {clean_protein_path}")

Cleaning receptor PDB for Smina compatibility...
SUCCESS -> Cleaned receptor saved to: ./t018_pipeline_output/3ika_clean.pdb


In [71]:
import os
import requests
from urllib.parse import quote
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess

output_dir = "./t018_pipeline_output"
os.makedirs(output_dir, exist_ok=True)
analog_sdf_path = os.path.join(output_dir, "analogs_library.sdf")
docked_output_path = os.path.join(output_dir, "docked_analogs.sdf")
protein_path = os.path.join(output_dir, "3ika_clean.pdb")

smiles = "COc1cc(ccc1Nc2ncc(Cl)c(Oc3cccc(NC(=O)C=C)c3)n2)N4CCN(C)CC4"

# 1. Ensure analogs_library.sdf is populated (using your lead SMILES + a few generated variants if PubChem lags)
print("Populating analog library...")
writer = Chem.SDWriter(analog_sdf_path)

# Add your primary lead molecule first
mol = Chem.MolFromSmiles(smiles)
if mol is not None:
  mol = Chem.AddHs(mol)
  AllChem.EmbedMolecule(mol, randomSeed=42)
  AllChem.MMFFOptimizeMolecule(mol)
  mol.SetProp("_Name", "Lead_Candidate")
  writer.write(mol)

# Try fetching a couple of CIDs from PubChem to add to the library
try:
  escaped_smiles = quote(smiles, safe="")
  url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/similarity/smiles/{escaped_smiles}/cids/JSON?Threshold=85&MaxRecords=3"
  res = requests.get(url).json()
  if "IdentifierList" in res:
    for cid in res["IdentifierList"]["CID"]:
      sdf_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF"
      sdf_data = requests.get(sdf_url).text
      m = Chem.MolFromMolBlock(sdf_data, removeHs=False)
      if m is not None:
        m = Chem.AddHs(m)
        AllChem.EmbedMolecule(m, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(m)
        m.SetProp("_Name", f"CID_{cid}")
        writer.write(m)
except Exception as e:
  print(f"PubChem lookup skipped/failed ({e}), proceeding with lead molecule.")

writer.close()
print(f"SUCCESS -> Analog library written to {analog_sdf_path}")

# 2. Run Smina Docking
box_center = [15.5, 32.1, 12.4]
box_size = [20.0, 20.0, 20.0]

smina_cmd = [
    "smina",
    "-r", protein_path,
    "-l", analog_sdf_path,
    "--center_x", str(box_center[0]),
    "--center_y", str(box_center[1]),
    "--center_z", str(box_center[2]),
    "--size_x", str(box_size[0]),
    "--size_y", str(box_size[1]),
    "--size_z", str(box_size[2]),
    "--exhaustiveness", "8",
    "-o", docked_output_path
]

print("Running Smina docking batch...")
result = subprocess.run(smina_cmd, capture_output=True, text=True)

if result.returncode == 0 and os.path.exists(docked_output_path) and os.path.getsize(docked_output_path) > 0:
  print("SUCCESS -> Docking complete and output file generated!")
  
  # 3. Parse and rank results
  supplier = Chem.SDMolSupplier(docked_output_path)
  results = []
  for i, m in enumerate(supplier):
    if m is not None:
      name = m.GetProp("_Name") if m.HasProp("_Name") else f"Molecule_{i+1}"
      score = float(m.GetProp("minimizedAffinity")) if m.HasProp("minimizedAffinity") else 0.0
      results.append({"Name": name, "Score": score, "Molecule": m})

  results = sorted(results, key=lambda x: x["Score"])
  print("\n--- Final T018 Ranking ---")
  for rank, r in enumerate(results, 1):
    print(f"Rank {rank}: {r['Name']} -> Affinity: {r['Score']} kcal/mol")
else:
  print("Docking failed or output file is empty.")
  print("STDERR:", result.stderr)

Populating analog library...
SUCCESS -> Analog library written to ./t018_pipeline_output/analogs_library.sdf
Running Smina docking batch...
SUCCESS -> Docking complete and output file generated!

--- Final T018 Ranking ---
Rank 1: Lead_Candidate -> Affinity: -7.85679 kcal/mol
Rank 2: Lead_Candidate -> Affinity: -7.63929 kcal/mol
Rank 3: Lead_Candidate -> Affinity: -7.45913 kcal/mol
Rank 4: Lead_Candidate -> Affinity: -7.4507 kcal/mol
Rank 5: Lead_Candidate -> Affinity: -7.36167 kcal/mol
Rank 6: Lead_Candidate -> Affinity: -7.23568 kcal/mol
Rank 7: Lead_Candidate -> Affinity: -7.20943 kcal/mol
Rank 8: Lead_Candidate -> Affinity: -7.13611 kcal/mol
Rank 9: Lead_Candidate -> Affinity: -7.11617 kcal/mol


[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:50:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol